In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Load the CSV file
Q1_data_path = os.path.join(path, 'Q1_data.csv')
Q1_data = pd.read_csv(Q1_data_path)

In [ ]:
Q1_data.head()

In [ ]:
Q1_data.info()

In [ ]:
Q1_data.describe()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(Q1_data["Delivery_Time"].dropna(), bins=30, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
data_clean = Q1_data.drop("Order_ID" , axis = 1)
data_clean

In [ ]:
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    data_clean[col] = data_clean[col].fillna('unknown')

for col in ['Courier_Experience_yrs', "Courier_Experience_yrs"]:
    data_clean[col] = data_clean[col].fillna(data_clean[col].mean())
data_clean = data_clean.dropna(subset = ["Delivery_Time"])
data_clean.isna().sum()

In [ ]:
print("Checking for duplicate rows...")
duplicate_rows = data_clean.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    data_clean.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

In [ ]:
categories = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']

ct = ColumnTransformer(
    transformers = [
        ("one hot encoder" , OneHotEncoder(handle_unknown="ignore" ), categories), # column transformer handles adding the columns and every thing automatticly so i used it because i used to work with it forggot how to oneHot (:
    ],
    remainder = "passthrough"
)


y = data_clean["Delivery_Time"]
X = ct.fit_transform(data_clean.drop("Delivery_Time", axis = 1))


In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [ ]:
# from the above graph it looks balanced a little skewed i guess, but i will say its balaced for sure also the 150 data could be outliers

In [ ]:

X_train , X_test , y_train , y_test = train_test_split(X, y, test_size = 0.1, random_state = 42)

In [ ]:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metric
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
features = [ f"feature {i}" for i in range(len(model.feature_importances_)) ]

# Feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred")
plt.title("Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: